# Upgrading `pyiron_workflow` to use `flowrep`


In version <=0.17.0, `pyiron_workflow` managed everything about workflows by holding everything in state at once: the recipe, the result, and the configuration to apply computational resources to generate results from the recipe. In version 0.18.0, we switched over to an implementation based on `flowrep`.

In this flowrep-based approach, prospective recipes are defined as pydantic models in flowrep, while retrospective results are dataclasses in flowrep. A `pyiron_workflow` run converts a recipe into a `Run` object, which contains the flowrep retrospective data object -- note that this differs from what the old implementation produces, but a simple port label-keyed dictionary of result data is still easily available under the `.outputs` attribute of the returned `Run` instance. The remaining responsibilities of `pyiron_workflow` are (a) to apply computational resources to convert a recipe into a result, and (b) to help users dynamically construct new workflows.

The user-facing differences between the old- and new-implementations of `pyiron_workflow` are not extreme, and we hope you'll find the new approach to be overall more useful and powerful. While the set of features is largely overlapping, there are some syntax differences. In this notebook, we'll walk through the process of taking some nominal legacy-syntax node library and workflow, and following the advice of any warnings or error messages we get to upgrade it.

In [1]:
from concurrent import futures

import pydantic
import pyiron_workflow as pwf
import flowrep as fr


class MyData:
    def __init__(self):
        self.data = 42

## Part 1 -- The Library

First, let's look at building a library of function and macro nodes for some toy problem. We'll kick things off with a simple function node

In [3]:
try:
    @pwf.as_function_node("my_sum", use_cache=False)
    def my_sum(x, y):
        return x + y
except ValueError as e:
    print(e)

Compatibility decorators take legacy-decorated functions, and turn them into factories for new node classes. In this context, arguments (other than output labels) to the decorator are not meaningful. Can't parse 'dispatch_output_labels.<locals>.multi_dispatch_decorator' as a compatibility factory because it received kwargs {'use_cache': False}


The message calls it a "compatibility decorator" because `pyiron_workflow` itself doesn't actually have node decorators any more --  we rely on `flowrep` directly for turning python code into node recipes. We'll see that in the final section. In the meantime, the immediate error is caused by `use_cache=False`. Caching is no longer managed directly by `pyiron_workflow` -- we recommend using `fleche`, which you can read about in the user guide. Indeed, none of the decorator-argument kwargs are valid any more. The output labels are still fine though, and function just as they always did. Let's fix it up and keep going.

In [4]:
@pwf.as_function_node("sum")
def my_sum(x, y):
    return x + y

Next, let's make a simple macro

In [5]:
@pwf.as_function_node
def process_data(data: MyData, norm: int | float) -> float:
    normalized_data = data.data / float(norm)
    return normalized_data


try:
    @pwf.as_macro_node
    def my_macro(self, x, rescaled):
        self.iterate = my_sum(x, 1)
        self.normalized = process_data()
        self.normalized.inputs.data = MyData()
        self.normalized.inputs.norm = self.iterate
        self.scaled = self.normalized * rescaled
        return self.scaled
except ValueError as e:
    print(e)

Flowrep-based pyiron_workflow requires that data be passed to nodes using keyword arguments, but 'my_sum' received the args (InputPort(label='x', owner=<pyiron_workflow.workflow_node.Workflow object at 0x106c03560>, type_hint=None, type_metadata=None, has_default=False), 1). Please use keywords among ['x', 'y'].


The first thing we see is simple enough --  we can't use args any more to pass incoming connection data, we need to use kwargs

In [6]:
try:
    @pwf.as_macro_node
    def my_macro(self, x, rescaled):
        self.iterate = my_sum(x=x, y=1)
        self.normalized = process_data()
        self.normalized.inputs.data = MyData()
        self.normalized.inputs.norm = self.iterate
        self.scaled = self.normalized * rescaled
        return self.scaled
except TypeError as e:
    print(e)

Cannot use <__main__.MyData object at 0x121131580> as the source for input 'data' of 'my_macro.normalized': expected a Port, a Node, or a JSONable constant. 
Older versions of pyiron_workflow allowed arbitrary python objects to be assigned as input port data; with the Flowrep-based approach, non-JSONable data needs to be sourced by edges to the terminal input of the workflow, or come from a bespoke node that produces that complex data. To recover pre-flowrep pyiron_workflow, downgrade to version <=0.17.0, e.g. with `conda install -c conda-forge pyiron_workflow=0.17.0`


The next thing we see is that we can't hard-code the instance of our custom-class `MyData()` as input to a node. Now, there's nothing wrong with using the `...inputs.{PORT_NAME} = ...` syntax, it works just as well as kwargs at initialization time, and there's nothing wrong with _JSONable_ constants (str, int, float, None, and lists and dicts of these) -- indeed, the `y=1` when we declared `self.iterate = my_sum(x=x, y=1)` is doing exactly this sort of hard coding. Under the hood, it injects a new (JSONable) constant node into the recipe. _But_, we can't pull this trick with _non_-JSONable data.

We'll follow the advice of the error message and promote the value to be sourced from the macro input:

In [7]:
@pwf.as_macro_node
def my_macro(self, x, rescaled, my_data):
    self.iterate = my_sum(x=x, y=1)
    self.normalized = process_data()
    self.normalized.inputs.data = my_data
    self.normalized.inputs.norm = self.iterate
    self.scaled = self.normalized * rescaled
    return self.scaled

While `flowrep` doesn't support injection on operators, `pyiron_workflow` still does, so we can carry on using the `self.normalized * rescaled` line just like always. And thus, with a little massaging, our old-style macro definition is now a creator for a new-style node. New-style nodes store their `flowrep` recipes, and we can look at this one:

In [8]:
macro_instance = my_macro()
print(macro_instance.recipe.model_dump_json(indent=2))

{
  "type": "workflow",
  "inputs": [
    "x",
    "rescaled",
    "my_data"
  ],
  "outputs": [
    "scaled"
  ],
  "description": null,
  "nodes": {
    "my_sum_0": {
      "type": "atomic",
      "inputs": [
        "x",
        "y"
      ],
      "outputs": [
        "sum"
      ],
      "description": null,
      "reference": {
        "info": {
          "module": "__main__",
          "qualname": "my_sum.decorated",
          "version": null
        },
        "inputs_with_defaults": [],
        "restricted_input_kinds": {}
      }
    },
    "constant_0": {
      "type": "constant",
      "inputs": [],
      "outputs": [
        "constant"
      ],
      "description": null,
      "constant": 1
    },
    "process_data_0": {
      "type": "atomic",
      "inputs": [
        "data",
        "norm"
      ],
      "outputs": [
        "normalized_data"
      ],
      "description": null,
      "reference": {
        "info": {
          "module": "__main__",
          "qualname": "

## Part 2 -- The Workflow

The `self` argument in `def my_macro` is already a new-style `Workflow` node, just as `my_sum(...)` returned a new style `Atomic` (aka "function") node, so we've already seen most of the syntactic differences in terms of defining recipes. In this section, we'll look at the similarities and differences for spinning up a new user-mutatable workflow and applying computational resources to get a result from it.

In [9]:
try:
    wf = pwf.Workflow("demo", my_macro())
except TypeError as e:
    print(e)

`Workflow` takes positional `label: str` and (optional) `undo_limit: int = 10` arguments, but in addition to the label, received [<pyiron_workflow.dag.Macro object at 0x30c819550>].

In version <=0.17.0, the `Workflow` class took child nodes as positional arguments. If this was your intent, please read the new readme at: https://pyiron-workflow.readthedocs.io/en/latest/source/notebooks/user_guide.html

To recover pre-flowrep pyiron_workflow, downgrade to version <=0.17.0, e.g. with `conda install -c conda-forge pyiron_workflow=0.17.0`


The first thing we see is that we can't pre-populate workflows with node instances. No worries, we already saw in our macro definition that we can easily add nodes via attribute assignment. We also know that we can create new edges via kwargs. Putting this together, we can spin up a workflow and run it:

In [10]:
wf = pwf.Workflow("demo")
wf.macro = my_macro()
wf.deprecate_rescaled = my_sum(x=wf.macro, y=-1)

try:
    run = wf.run()
except pydantic.ValidationError as e:
    print(e)

1 validation error for WorkflowRecipe
  Value error, Could not find a source or default for the target: macro.x [type=value_error, input_value={'inputs': [], 'outputs':...')}, 'output_edges': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


/var/folders/nn/6kd6nhmj2rx7610kd9r_8h0m0000gn/T/ipykernel_92905/3247062052.py:6: UserWarning: 'demo' has no input ports, so running it takes no data. Older versions of pyiron_workflow exposed every unfed child input automatically; call `set_inputs_to_unconnected_child_input` to reproduce that behaviour, or build the input you want with `create_input` and/or `create_input_for`.

To recover pre-flowrep pyiron_workflow, downgrade to version <=0.17.0, e.g. with `conda install -c conda-forge pyiron_workflow=0.17.0`
  run = wf.run()
/var/folders/nn/6kd6nhmj2rx7610kd9r_8h0m0000gn/T/ipykernel_92905/3247062052.py:6: UserWarning: 'demo' has no output ports, so running it produces no data. Older versions of pyiron_workflow exposed every unconnected child output automatically; call `set_outputs_to_unconnected_child_output` to reproduce that behaviour, or build the output you want with `create_output` and/or `create_output_from`.

To recover pre-flowrep pyiron_workflow, downgrade to version <=0.17

The proximate error is that our user-mutable workflow failed to produce a valid `flowrep` recipe -- in particular, because `macro.x` needed input and didn't have any. In addition to this, we see some warnings that shed additional light: legacy `pyiron_workflow` auto-generated IO for our workflows from all the dangling child IO; now workflow IO is more explicit. We'll take the suggested shortcut:

In [11]:
wf.set_inputs_to_unconnected_child_input()

run = wf.run()
run.outputs

/var/folders/nn/6kd6nhmj2rx7610kd9r_8h0m0000gn/T/ipykernel_92905/2650812753.py:3: UserWarning: 'demo' has no output ports, so running it produces no data. Older versions of pyiron_workflow exposed every unconnected child output automatically; call `set_outputs_to_unconnected_child_output` to reproduce that behaviour, or build the output you want with `create_output` and/or `create_output_from`.

To recover pre-flowrep pyiron_workflow, downgrade to version <=0.17.0, e.g. with `conda install -c conda-forge pyiron_workflow=0.17.0`
  run = wf.run()


{}

This is didn't fail, but we see that we didn't get any actual result. The warning lets us no why: we got the workflow to feed input to its children, but no output edges exist yet. We could fix this with the recommended patch, but instead we'll use a combo method to set input and output at once

In [12]:
wf.set_io_to_unconnected_child_io()

run = wf.run()
run.outputs

{'deprecate_rescaled__sum': NOT_DATA}

Alright, we're almost there! We get no errors and no warnings, but also no data! Making this moment noisier [is an open issue](https://github.com/pyiron/pyiron_workflow/issues/913), but the problem is simply that we forgot to feed the workflow any input data. Let's take a quick peek at what our input is named and then fill it up.

In [13]:
print(wf.inputs)

MutableInputMap(demo): 'macro__x': InputPort(demo.inputs.macro__x)
  'macro__rescaled': InputPort(demo.inputs.macro__rescaled)
  'macro__my_data': InputPort(demo.inputs.macro__my_data)



In [14]:
run = wf.run(macro__x=6, macro__rescaled=5, macro__my_data=MyData())
run.outputs  # ((42 / (x + 1)) * rescaled) - 1

{'deprecate_rescaled__sum': 29.0}

Perfect. Finally, let's apply some (unnecessary) compute power. Applying executor _instances_ looks just like it always did, but executor _instructions_ look a little different. In the legacy implementation we'd provide a (constructor, args, kwargs) tuple:

In [15]:
try:
    wf.deprecate_rescaled.executor = (futures.ThreadPoolExecutor, (), {})
    run = wf.run(macro__x=1, macro__rescaled=1, macro__my_data=MyData())
    run.outputs
except TypeError as e:
    print(e)

Expected executor to be an instance of ExecutorInstructions or futures.Executor, but 'demo.deprecate_rescaled' got (<class 'concurrent.futures.thread.ThreadPoolExecutor'>, (), {}).


Per the error message, this tuple has been formalized into a little data class. The args and kwargs have defaults here, so all we need right now is the constructor.

In [16]:
wf.deprecate_rescaled.executor = pwf.ExecutorInstructions(futures.ThreadPoolExecutor)
run = wf.run(macro__x=1, macro__rescaled=1, macro__my_data=MyData())
run.outputs

{'deprecate_rescaled__sum': 20.0}

What we've seen here for creating and mutating workflows is the syntactic sugar that looks like the legacy implementation. For a full rundown of the `Workflow` API, go read the new, updated user guide notebook.

## Part 3 -- Flowrep-native

These compatibility decorators are deprecated -- eventually they'll be gone. That means that we want to eventually replace our `@as_function/macro_node` decorators with `@flowrep.atomic` and `@flowrep.workflow` decorators. Furthermore, there are some legacy items that have no equivalent; these will fail and give you some advice if you try to use them. E.g., let's look at upgrading a legacy macro definition using a while-loop by re-writing it directly as a `@flowrep.workflow`.

In [17]:
try:
    @pwf.as_macro_node
    def loop_macro(self, x_0, x_lim, dx=2):
        self.add_while = pwf.while_node(
            pwf.std.LessThan,  # Test
            pwf.std.Add,  # Body
            [("add", "obj")],  # body-to-test
            [("add", "obj")],  # body-to-body
            strict_condition_hint=False,  # LessThan doesn't hint it's boolean return...
            test_obj=x_0,
            test_other=x_lim,
            body_obj=x_0,
            body_other=dx
        )
        self.xf = wf.add_while.outputs.add  # While output maps to body output
        self.iterated = my_sum(x=self.xf, y=1)
        return self.iterated
except pwf.compatibility.RemovedFeatureError as e:
    print(e)

while_node was removed in version 0.19.0; The most recent version retaining this object as the primary interface is version 0.17.0.

In the flowrep-based paradigm:
	In the context of parsing a `@flowrep.workflow`-decorated function, you can write while-loops with constrained but native python code -- see the flowrep user guide. Further, you can directly write a flowrep while-loop recipe with `flowrep.schemas.WhileRecipe` and turn that into a pyiron_workflow node, e.g. `wf.while_node = pyiron_workflow.node(my_while_recipe)`.

For more on flowrep: https://github.com/pyiron/flowrep
The flowrep user guide: https://flowrep.readthedocs.io/en/latest/source/notebooks/user-guide.html

To recover pre-flowrep pyiron_workflow, downgrade to version <=0.17.0, e.g. with `conda install -c conda-forge pyiron_workflow=0.17.0`


We'll need to switch over to the `flowrep` decorator and parser. That means no more `self` argument, because the decorated functions don't become node-constructors, they're just functions! Also, we have no more `pwf.std`:

In [19]:
try:
    from pyiron_workflow import std
except pwf.compatibility.RemovedFeatureError as e:
    print(e)

std was removed in version 0.19.0; The most recent version retaining this object as the primary interface is version 0.17.0.

In the flowrep-based paradigm:
	Use `flowrep.std` instead, and convert standard recipes to nodes when adding them to a workflow, e.g. `wf.add = pyiron_workflow.node(flowrep.std.add)`.

For more on flowrep: https://github.com/pyiron/flowrep
The flowrep user guide: https://flowrep.readthedocs.io/en/latest/source/notebooks/user-guide.html

To recover pre-flowrep pyiron_workflow, downgrade to version <=0.17.0, e.g. with `conda install -c conda-forge pyiron_workflow=0.17.0`


Putting this together, the same workflow written in `flowrep` looks much more like the native python -- and indeed, the whole workflow is still just a native python function! Indeed, calling the function gives you the exact same result as we store in the output port of our workflow run result:

In [20]:
@fr.workflow
def loop_macro(x_0, x_lim, dx=2):
    x = x_0
    while fr.std.lt(x, x_lim):
        x = fr.std.add(x, dx)
    return x

n = pwf.node(loop_macro.flowrep_recipe)
(
    n.run(x_0=1, x_lim=10).result.output_ports["x"].value,
    loop_macro(x_0=1, x_lim=10)
)

(11, 11)